# Calibrated Explanations: Factual SHAP Explanation Plugin Example

This notebook uses the installed SHAP explanation plugin through the public `WrapCalibratedExplainer` workflow.
It assumes the package `calibrated-explanations-explanation-factual-shap` is already installed in the current environment.

In [1]:
import os

import calibrated_explanations.plugins.registry as registry

EXPLANATION_ID = "official.explanation.factual.shap"

# Trust and discover installed plugins via package entry points.
os.environ["CE_TRUST_PLUGIN"] = EXPLANATION_ID
registry.load_entrypoint_plugins(include_untrusted=False)

desc = registry.find_explanation_descriptor(EXPLANATION_ID)
print("Explanation plugin loaded:", desc is not None)
print("Explanation plugin trusted:", getattr(desc, "trusted", None))

Explanation plugin loaded: True
Explanation plugin trusted: False


C:\Users\loftuw\CUDATemp\ipykernel_40596\978633600.py:9: UserWarning: Skipping untrusted plugin 'official.explanation.factual.shap' from official discovered via entrypoint. Set CE_TRUST_PLUGIN, add it to [tool.calibrated_explanations.plugins].trusted, or call trust_plugin('official.explanation.factual.shap') to load it.
  registry.load_entrypoint_plugins(include_untrusted=False)
C:\Users\loftuw\CUDATemp\ipykernel_40596\978633600.py:9: UserWarning: Skipping untrusted plugin 'official.visualization.factual.shap.bootstrap' from official discovered via entrypoint. Set CE_TRUST_PLUGIN, add it to [tool.calibrated_explanations.plugins].trusted, or call trust_plugin('official.visualization.factual.shap.bootstrap') to load it.
  registry.load_entrypoint_plugins(include_untrusted=False)


## Fit and Calibrate a Wrapped Model

Use the normal `WrapCalibratedExplainer` flow and select the installed SHAP explanation plugin during calibration.

In [2]:
from calibrated_explanations import WrapCalibratedExplainer
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X, y = make_classification(
    n_samples=240,
    n_features=6,
    n_informative=4,
    n_redundant=0,
    random_state=0,
)

X_train, X_test, y_train, _ = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=0,
    stratify=y,
)
X_fit, X_cal, y_fit, y_cal = train_test_split(
    X_train,
    y_train,
    test_size=0.25,
    random_state=0,
    stratify=y_train,
)

model = WrapCalibratedExplainer(LogisticRegression(random_state=0, solver="liblinear"))
model.fit(X_fit, y_fit)
model.calibrate(X_cal, y_cal, mode="classification", factual_plugin=EXPLANATION_ID)
model

WrapCalibratedExplainer(learner=LogisticRegression(random_state=0, solver='liblinear'), fitted=True, calibrated=True, 
		explainer=CalibratedExplainer(mode=classification, learner=LogisticRegression(random_state=0, solver='liblinear')))

## Generate SHAP-backed Factual Explanations

The SHAP explanation plugin replaces condition-style rule text with feature names and exposes bound-aware SHAP metadata.

In [3]:
instances = X_test[:5]
explanations = model.explain_factual(instances)

rules0 = explanations.explanations[0].get_rules()
shap_meta = explanations.batch_metadata.get("shap", {})

print("First five rule labels:", rules0["rule"][:5])
print("Contains condition operators?:", any("<" in label or ">" in label or "=" in label for label in rules0["rule"]))
print("Has lower/upper weights?:", "weight_low" in rules0 and "weight_high" in rules0)
print("Available SHAP value bounds:", sorted(shap_meta.get("values", {}).keys()))
print("Available SHAP base-value bounds:", sorted(shap_meta.get("base_values", {}).keys()))
print("SHAP data matrix shape:", __import__("numpy").asarray(shap_meta.get("data", [])).shape)

C:\Users\loftuw\Documents\Github\kristinebergs-calibrated_explanations\src\calibrated_explanations\core\explain\orchestrator.py:1790: UserWarning: Using untrusted explanation plugin 'official.explanation.factual.shap' via explicit override. Ensure you trust the source of this plugin.
  plugin, identifier = self.explainer.plugin_manager.resolve_explanation_plugin_for_mode(
ExactExplainer explainer: 6it [00:13,  2.74s/it]               

First five rule labels: ['0', '1', '2', '4', '5']
Contains condition operators?: False
Has lower/upper weights?: True
Available SHAP value bounds: ['center', 'lower', 'uncertainty', 'upper']
Available SHAP base-value bounds: ['center', 'lower', 'uncertainty', 'upper']
SHAP data matrix shape: (5, 6)
